In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from data_pipeline import cleaning_data

In [3]:
df=pd.read_csv(r"C:\Users\thien\code\AIMY\house-prices-advanced-regression-techniques\train.csv")
target_name='SalePrice'
drop_threshold=0.6

In [4]:
df=df.drop(columns='Id')

In [5]:
X,y=cleaning_data(df,target_name=target_name,drop_threshold=drop_threshold)

In [6]:
X.isna().any(axis=0).sum()

np.int64(0)

In [7]:
X.columns

Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'LotShape',
       'LandContour', 'LotConfig', 'Neighborhood', 'Condition1', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinSF2',
       'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir',
       'Electrical', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath',
       'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr',
       'KitchenQual', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageFinish', 'GarageCars', 'GarageQual', 'GarageCond', 'PavedDrive',
       'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch',
       'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'SaleType',
       'SaleC

In [8]:
from pandasql import sqldf

analys_sql

In [9]:
# ── Hàm tiện ích để chạy SQL ──────────────────────────────────────────────────
pysqldf = lambda q: sqldf(q, globals())

In [10]:
# ==============================================================================
# CÂU 1 – Thống kê tổng quan (trung bình, min, max) theo chất lượng tổng thể
# ==============================================================================
# Mục đích : Xem giá nhà (log) biến động như thế nào theo từng mức OverallQual.
#            Đây là feature có tương quan mạnh nhất với SalePrice.
# Kết quả  : OverallQual càng cao → avg_price càng lớn, khoảng dao động càng rộng.
q1 = pysqldf("""
    SELECT
        OverallQual,
        COUNT(*)                         AS so_nha,
        ROUND(AVG(SalePrice), 4)         AS avg_log_price,
        ROUND(MIN(SalePrice), 4)         AS min_log_price,
        ROUND(MAX(SalePrice), 4)         AS max_log_price
    FROM df
    GROUP BY OverallQual
    ORDER BY OverallQual
""")
print("\n[Q1] Giá trung bình theo OverallQual:")
print(q1.to_string(index=False))



[Q1] Giá trung bình theo OverallQual:
 OverallQual  so_nha  avg_log_price  min_log_price  max_log_price
           1       2     50150.0000        39300.0        61000.0
           2       3     51770.3333        35311.0        60000.0
           3      20     87473.7500        37900.0       139600.0
           4     116    108420.6552        34900.0       256000.0
           5     397    133523.3476        55993.0       228950.0
           6     374    161603.0348        76000.0       277000.0
           7     319    207716.4232        82500.0       383970.0
           8     168    274735.5357       122000.0       538000.0
           9      43    367513.0233       239000.0       611657.0
          10      18    438588.3889       160000.0       755000.0


In [11]:
# ==============================================================================
# CÂU 2 – Top 5 Neighborhood có giá nhà trung bình cao nhất
# ==============================================================================
# Mục đích : Tìm khu vực đắt đất nhất để định giá theo vùng.
# Kết quả  : Các neighborhood thuộc vùng cao cấp như NridgHt, StoneBr thường dẫn đầu.
q2 = pysqldf("""
    SELECT
        Neighborhood,
        COUNT(*)                         AS so_nha,
        ROUND(AVG(SalePrice), 4)         AS avg_log_price
    FROM df
    GROUP BY Neighborhood
    ORDER BY avg_log_price DESC
    LIMIT 5
""")
print("\n[Q2] Top 5 Neighborhood đắt nhất:")
print(q2.to_string(index=False))


[Q2] Top 5 Neighborhood đắt nhất:
Neighborhood  so_nha  avg_log_price
     NoRidge      41    335295.3171
     NridgHt      77    316270.6234
     StoneBr      25    310499.0000
      Timber      38    242247.4474
     Veenker      11    238772.7273


In [12]:
# ==============================================================================
# CÂU 3 – Phân phối nhà theo số phòng ngủ (BedroomAbvGr) và loại nhà (BldgType)
# ==============================================================================
# Mục đích : Hiểu cơ cấu sản phẩm – loại nhà nào có nhiều phòng ngủ nhất.
# Kết quả  : Nhà 1 gia đình (1Fam) chiếm đa số, thường 3 phòng ngủ.
q3 = pysqldf("""
    SELECT
        BldgType,
        BedroomAbvGr,
        COUNT(*) AS so_nha
    FROM df
    GROUP BY BldgType, BedroomAbvGr
    ORDER BY BldgType, BedroomAbvGr
""")
print("\n[Q3] Phân phối nhà theo BldgType & BedroomAbvGr:")
print(q3.to_string(index=False))



[Q3] Phân phối nhà theo BldgType & BedroomAbvGr:
BldgType  BedroomAbvGr  so_nha
    1Fam             0       3
    1Fam             1      21
    1Fam             2     249
    1Fam             3     751
    1Fam             4     180
    1Fam             5      16
  2fmCon             2       7
  2fmCon             3      14
  2fmCon             4       6
  2fmCon             5       2
  2fmCon             6       1
  2fmCon             8       1
  Duplex             0       2
  Duplex             2      13
  Duplex             3       3
  Duplex             4      26
  Duplex             5       2
  Duplex             6       6
   Twnhs             1       5
   Twnhs             2      17
   Twnhs             3      21
  TwnhsE             0       1
  TwnhsE             1      24
  TwnhsE             2      72
  TwnhsE             3      15
  TwnhsE             4       1
  TwnhsE             5       1


In [13]:
# ==============================================================================
# CÂU 4 – Giá trung bình theo chất lượng bếp (KitchenQual)
# ==============================================================================
# Mục đích : Bếp chất lượng cao có ảnh hưởng đến giá không?
# Kết quả  : Ex > Gd > TA > Fa – chất lượng bếp tỉ lệ thuận rõ ràng với giá.
q4 = pysqldf("""
    SELECT
        KitchenQual,
        COUNT(*)                   AS so_nha,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price
    FROM df
    GROUP BY KitchenQual
    ORDER BY avg_log_price DESC
""")
print("\n[Q4] Giá trung bình theo KitchenQual:")
print(q4.to_string(index=False))


[Q4] Giá trung bình theo KitchenQual:
KitchenQual  so_nha  avg_log_price
         Ex     100    328554.6700
         Gd     586    212116.0239
         TA     735    139962.5116
         Fa      39    105565.2051


In [14]:
# ==============================================================================
# CÂU 5 – Nhà có điều hòa (CentralAir) vs không có – chênh lệch giá bao nhiêu?
# ==============================================================================
# Mục đích : Lượng hóa giá trị của hệ thống điều hòa trung tâm.
# Kết quả  : Nhà có CentralAir=Y thường cao hơn đáng kể so với N.
q5 = pysqldf("""
    SELECT
        CentralAir,
        COUNT(*)                   AS so_nha,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price,
        ROUND(MAX(SalePrice) - MIN(SalePrice), 4) AS range_price
    FROM df
    GROUP BY CentralAir
""")
print("\n[Q5] Ảnh hưởng CentralAir đến giá:")
print(q5.to_string(index=False))


[Q5] Ảnh hưởng CentralAir đến giá:
CentralAir  so_nha  avg_log_price  range_price
         N      95    105264.0737     231079.0
         Y    1365    186186.7099     703000.0


In [15]:
# ==============================================================================
# CÂU 6 – Giá trung bình theo số xe garage (GarageCars)
# ==============================================================================
# Mục đích : Garage lớn hơn thì giá có tăng tuyến tính không?
# Kết quả  : GarageCars = 3 thường cao nhất; = 0 thấp nhất.
q6 = pysqldf("""
    SELECT
        GarageCars,
        COUNT(*)                   AS so_nha,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price
    FROM df
    GROUP BY GarageCars
    ORDER BY GarageCars
""")
print("\n[Q6] Giá trung bình theo GarageCars:")
print(q6.to_string(index=False))


[Q6] Giá trung bình theo GarageCars:
 GarageCars  so_nha  avg_log_price
          0      81    103317.2840
          1     369    128116.6883
          2     824    183851.6638
          3     181    309636.1215
          4       5    192655.8000


In [16]:
# ==============================================================================
# CÂU 7 – Xu hướng giá nhà theo năm bán (YrSold) và tháng bán (MoSold)
# ==============================================================================
# Mục đích : Kiểm tra tính thời vụ – tháng nào bán đắt nhất?
# Kết quả  : Thường mùa hè (tháng 5-7) giá cao hơn mùa đông.
q7 = pysqldf("""
    SELECT
        YrSold,
        MoSold,
        COUNT(*)                   AS so_giao_dich,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price
    FROM df
    GROUP BY YrSold, MoSold
    ORDER BY YrSold, MoSold
""")
print("\n[Q7] Xu hướng giá theo YrSold & MoSold (5 dòng đầu):")
print(q7.head(10).to_string(index=False))



[Q7] Xu hướng giá theo YrSold & MoSold (5 dòng đầu):
 YrSold  MoSold  so_giao_dich  avg_log_price
   2006       1            10    201090.0000
   2006       2             9    194322.2222
   2006       3            25    184982.2000
   2006       4            27    174312.8148
   2006       5            38    158928.2895
   2006       6            48    172283.3333
   2006       7            67    183211.0597
   2006       8            23    196239.9565
   2006       9            15    223768.8667
   2006      10            24    172356.7083


In [ ]:
# ==============================================================================
# CÂU 8 – Tỉ lệ nhà được lát vỉa hè (PavedDrive) theo loại lô đất (LotConfig)
# ==============================================================================
# Mục đích : Lô đất góc phố hay trong ngõ thì hay có vỉa hè hơn không?
# Kết quả  : Lô Corner & CulDSac thường có PavedDrive=Y cao hơn.
q8 = pysqldf("""
    SELECT
        LotConfig,
        PavedDrive,
        COUNT(*) AS so_nha,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY LotConfig), 2) AS pct
    FROM df
    GROUP BY LotConfig, PavedDrive
    ORDER BY LotConfig, PavedDrive
""")
print("\n[Q8] Tỉ lệ PavedDrive theo LotConfig:")
print(q8.to_string(index=False))

In [17]:
# ==============================================================================
# CÂU 9 – Top 10 nhà có diện tích sống (GrLivArea) lớn nhất
# ==============================================================================
# Mục đích : Tìm những căn nhà rộng nhất và kiểm tra giá tương ứng.
# Kết quả  : GrLivArea lớn không nhất thiết là đắt nhất (outliers đã winsorize).
q9 = pysqldf("""
    SELECT
        rowid                      AS id,
        Neighborhood,
        HouseStyle,
        ROUND(GrLivArea, 0)        AS dien_tich_song,
        OverallQual,
        ROUND(SalePrice, 4)        AS log_price
    FROM df
    ORDER BY GrLivArea DESC
    LIMIT 10
""")
print("\n[Q9] Top 10 nhà có GrLivArea lớn nhất:")
print(q9.to_string(index=False))


[Q9] Top 10 nhà có GrLivArea lớn nhất:
  id Neighborhood HouseStyle  dien_tich_song  OverallQual  log_price
1299      Edwards     2Story          5642.0           10   160000.0
 524      Edwards     2Story          4676.0           10   184750.0
1183      NoRidge     2Story          4476.0           10   745000.0
 692      NoRidge     2Story          4316.0           10   755000.0
1170      NoRidge     2Story          3627.0           10   625000.0
 186      OldTown     2.5Fin          3608.0           10   475000.0
 305      OldTown     2.5Fin          3493.0            7   295000.0
1269      Crawfor     1.5Fin          3447.0            8   381000.0
 636        SWISU     2.5Fin          3395.0            6   200000.0
 770      StoneBr     2Story          3279.0            8   538000.0


In [18]:
# ==============================================================================
# CÂU 10 – Số nhà có lò sưởi (Fireplaces > 0) theo từng khu vực
# ==============================================================================
# Mục đích : Lò sưởi phổ biến ở khu vực nào? Liên quan đến khí hậu/đẳng cấp.
# Kết quả  : Khu cao cấp thường có nhiều nhà với lò sưởi hơn.
q10 = pysqldf("""
    SELECT
        Neighborhood,
        SUM(CASE WHEN Fireplaces > 0 THEN 1 ELSE 0 END) AS co_lo_suoi,
        SUM(CASE WHEN Fireplaces = 0 THEN 1 ELSE 0 END) AS khong_lo_suoi,
        COUNT(*)                                          AS tong
    FROM df
    GROUP BY Neighborhood
    ORDER BY co_lo_suoi DESC
    LIMIT 8
""")
print("\n[Q10] Số nhà có lò sưởi theo Neighborhood:")
print(q10.to_string(index=False))


[Q10] Số nhà có lò sưởi theo Neighborhood:
Neighborhood  co_lo_suoi  khong_lo_suoi  tong
       NAmes          97            128   225
     NridgHt          70              7    77
     Gilbert          67             12    79
      NWAmes          61             12    73
     CollgCr          57             93   150
     Crawfor          45              6    51
     Somerst          40             46    86
     NoRidge          40              1    41


In [19]:
# ==============================================================================
# CÂU 11 – Giá trung bình theo loại mái nhà (RoofStyle)
# ==============================================================================
# Mục đích : Kiểu mái có ảnh hưởng đến giá trị thẩm mỹ và giá không?
# Kết quả  : Mái Hip thường cao hơn mái Gable phổ thông.
q11 = pysqldf("""
    SELECT
        RoofStyle,
        COUNT(*)                   AS so_nha,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price
    FROM df
    GROUP BY RoofStyle
    ORDER BY avg_log_price DESC
""")
print("\n[Q11] Giá trung bình theo RoofStyle:")
print(q11.to_string(index=False))


[Q11] Giá trung bình theo RoofStyle:
RoofStyle  so_nha  avg_log_price
     Shed       2    225000.0000
      Hip     286    218876.9336
     Flat      13    194690.0000
  Mansard       7    180568.4286
    Gable    1141    171483.9562
  Gambrel      11    148909.0909


In [20]:
# ==============================================================================
# CÂU 12 – Phân tích tầng hầm: tổng diện tích tầng hầm (TotalBsmtSF) theo BsmtQual
# ==============================================================================
# Mục đích : Chất lượng tầng hầm có tương quan với diện tích tầng hầm không?
# Kết quả  : BsmtQual=Ex thường có TotalBsmtSF lớn nhất.
q12 = pysqldf("""
    SELECT
        BsmtQual,
        COUNT(*)                       AS so_nha,
        ROUND(AVG(TotalBsmtSF), 2)     AS avg_bsmt_sf,
        ROUND(AVG(SalePrice), 4)       AS avg_log_price
    FROM df
    GROUP BY BsmtQual
    ORDER BY avg_log_price DESC
""")
print("\n[Q12] Tầng hầm theo BsmtQual:")
print(q12.to_string(index=False))


[Q12] Tầng hầm theo BsmtQual:
BsmtQual  so_nha  avg_bsmt_sf  avg_log_price
      Ex     121      1645.60    327041.0413
      Gd     618      1131.07    202688.4790
      TA     649       955.43    140759.8182
      Fa      35       733.03    115692.0286
     NaN      37         0.00    105652.8919


In [21]:
# ==============================================================================
# CÂU 13 – Nhà xây trước 1980 vs sau 1980: giá và chất lượng trung bình
# ==============================================================================
# Mục đích : Nhà cũ vs nhà mới khác nhau như thế nào về giá và chất lượng?
# Kết quả  : Nhà sau 1980 thường có OverallQual và giá cao hơn.
q13 = pysqldf("""
    SELECT
        CASE WHEN YearBuilt < 1980 THEN 'Truoc 1980' ELSE 'Tu 1980 tro di' END AS thoi_ky,
        COUNT(*)                     AS so_nha,
        ROUND(AVG(OverallQual), 2)   AS avg_qual,
        ROUND(AVG(SalePrice), 4)     AS avg_log_price
    FROM df
    GROUP BY thoi_ky
""")
print("\n[Q13] Nhà cũ vs nhà mới:")
print(q13.to_string(index=False))


[Q13] Nhà cũ vs nhà mới:
       thoi_ky  so_nha  avg_qual  avg_log_price
    Truoc 1980     848      5.33    142987.9281
Tu 1980 tro di     612      7.16    233482.3252


In [22]:
# ==============================================================================
# CÂU 14 – Diện tích boong gỗ (WoodDeckSF) vs hiên thông thoáng (OpenPorchSF): nhà nào có cả hai?
# ==============================================================================
# Mục đích : Tìm các nhà "outdoor friendly" – có cả deck lẫn porch.
# Kết quả  : Những nhà này thường có giá và diện tích sống cao hơn mức trung bình.
q14 = pysqldf("""
    SELECT
        CASE
            WHEN WoodDeckSF > 0 AND OpenPorchSF > 0 THEN 'Co ca hai'
            WHEN WoodDeckSF > 0                     THEN 'Chi co Deck'
            WHEN OpenPorchSF > 0                    THEN 'Chi co Porch'
            ELSE 'Khong co gi'
        END AS loai_ngoai_troi,
        COUNT(*)                   AS so_nha,
        ROUND(AVG(GrLivArea), 2)   AS avg_dien_tich_song,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price
    FROM df
    GROUP BY loai_ngoai_troi
    ORDER BY avg_log_price DESC
""")
print("\n[Q14] Phân tích WoodDeck & OpenPorch:")
print(q14.to_string(index=False))


[Q14] Phân tích WoodDeck & OpenPorch:
loai_ngoai_troi  so_nha  avg_dien_tich_song  avg_log_price
      Co ca hai     437             1779.14    234685.7208
   Chi co Porch     367             1554.50    181798.8610
    Chi co Deck     262             1364.32    157049.9695
    Khong co gi     394             1287.15    136345.2081


In [23]:
# ==============================================================================
# CÂU 15 – Phân tích loại hình bán hàng (SaleType) và điều kiện bán (SaleCondition)
# ==============================================================================
# Mục đích : Giao dịch "Normal" so với "Partial" (nhà mới chưa hoàn thiện) khác giá bao nhiêu?
# Kết quả  : SaleCondition=Partial thường cao hơn do nhà mới (New build).
q15 = pysqldf("""
    SELECT
        SaleType,
        SaleCondition,
        COUNT(*)                   AS so_giao_dich,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price
    FROM df
    GROUP BY SaleType, SaleCondition
    ORDER BY avg_log_price DESC
    LIMIT 10
""")
print("\n[Q15] Giá theo SaleType & SaleCondition:")
print(q15.to_string(index=False))


[Q15] Giá theo SaleType & SaleCondition:
SaleType SaleCondition  so_giao_dich  avg_log_price
     CWD       Abnorml             1    328900.0000
     New       Partial           122    274945.4180
     Con        Normal             2    269600.0000
   ConLD       Partial             1    235128.0000
   ConLI        Normal             4    219237.5000
     CWD        Normal             2    188750.0000
      WD        Normal          1160    175714.2750
      WD        Alloca            12    167377.4167
      WD        Family            19    150315.7895
      WD       Abnorml            70    147607.7000


In [24]:
# ==============================================================================
# CÂU 16 – Nhà có pool (PoolArea > 0): đặc điểm và giá
# ==============================================================================
# Mục đích : Bể bơi đóng góp bao nhiêu vào giá trị căn nhà?
# Kết quả  : Nhà có bể bơi rất ít (~1%) nhưng giá trung bình cao hơn đáng kể.
q16 = pysqldf("""
    SELECT
        CASE WHEN PoolArea > 0 THEN 'Co be boi' ELSE 'Khong co' END AS co_be_boi,
        COUNT(*)                   AS so_nha,
        ROUND(AVG(PoolArea), 2)    AS avg_pool_area,
        ROUND(AVG(OverallQual), 2) AS avg_qual,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price
    FROM df
    GROUP BY co_be_boi
""")
print("\n[Q16] Nhà có bể bơi vs không:")
print(q16.to_string(index=False))



[Q16] Nhà có bể bơi vs không:
co_be_boi  so_nha  avg_pool_area  avg_qual  avg_log_price
Co be boi       7         575.43      7.57    288138.5714
 Khong co    1453           0.00      6.09    180404.6635


In [25]:
# ==============================================================================
# CÂU 17 – Số phòng tắm đầy đủ (FullBath + BsmtFullBath) và tác động lên giá
# ==============================================================================
# Mục đích : Tổng số phòng tắm (kể cả tầng hầm) ảnh hưởng thế nào đến giá?
# Kết quả  : Nhà có 3+ phòng tắm giá tăng rõ rệt.
q17 = pysqldf("""
    SELECT
        (FullBath + BsmtFullBath)  AS tong_phong_tam,
        COUNT(*)                   AS so_nha,
        ROUND(AVG(GrLivArea), 2)   AS avg_dien_tich_song,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price
    FROM df
    GROUP BY tong_phong_tam
    ORDER BY tong_phong_tam
""")
print("\n[Q17] Giá theo tổng số phòng tắm đầy đủ:")
print(q17.to_string(index=False))



[Q17] Giá theo tổng số phòng tắm đầy đủ:
 tong_phong_tam  so_nha  avg_dien_tich_song  avg_log_price
              0       1             1402.00    194201.0000
              1     371             1186.04    124160.1105
              2     750             1499.11    174381.9507
              3     319             1863.58    254481.1505
              4      18             2841.28    319021.8889
              6       1             1200.00    179000.0000


In [26]:
# ==============================================================================
# CÂU 18 – Nhà được cải tạo (YearRemodAdd > YearBuilt) vs chưa cải tạo
# ==============================================================================
# Mục đích : Cải tạo có làm tăng giá so với nhà nguyên gốc không?
# Kết quả  : Nhà đã cải tạo thường có OverallCond và giá cao hơn.
q18 = pysqldf("""
    SELECT
        CASE WHEN YearRemodAdd > YearBuilt THEN 'Da cai tao' ELSE 'Chua cai tao' END AS tinh_trang,
        COUNT(*)                     AS so_nha,
        ROUND(AVG(OverallCond), 2)   AS avg_cond,
        ROUND(AVG(SalePrice), 4)     AS avg_log_price
    FROM df
    GROUP BY tinh_trang
""")
print("\n[Q18] Nhà đã cải tạo vs chưa cải tạo:")
print(q18.to_string(index=False))



[Q18] Nhà đã cải tạo vs chưa cải tạo:
  tinh_trang  so_nha  avg_cond  avg_log_price
Chua cai tao     764      5.25    182583.6597
  Da cai tao     696      5.94    179096.3075


In [27]:
# ==============================================================================
# CÂU 19 – Phân vị giá (quartile) theo kiểu nhà (HouseStyle)
# ==============================================================================
# Mục đích : Phân tích phân phối giá trong từng kiểu nhà (1 tầng, 2 tầng...).
# Kết quả  : Nhà 2 tầng (2Story) thường có median và Q3 cao hơn nhà 1 tầng.
q19 = pysqldf("""
    SELECT
        HouseStyle,
        COUNT(*)                          AS so_nha,
        ROUND(MIN(SalePrice), 4)          AS min_price,
        ROUND(AVG(SalePrice), 4)          AS avg_price,
        ROUND(MAX(SalePrice), 4)          AS max_price
    FROM df
    GROUP BY HouseStyle
    ORDER BY avg_price DESC
""")
print("\n[Q19] Phân vị giá theo HouseStyle:")
print(q19.to_string(index=False))


[Q19] Phân vị giá theo HouseStyle:
HouseStyle  so_nha  min_price   avg_price  max_price
    2.5Fin       8   104000.0 220000.0000   475000.0
    2Story     445    40000.0 210051.7640   755000.0
    1Story     726    34900.0 175985.4780   611657.0
      SLvl      65    91000.0 166703.3846   345000.0
    2.5Unf      11   101000.0 157354.5455   325000.0
    1.5Fin     154    37900.0 143116.7403   410000.0
    SFoyer      37    75500.0 135074.4865   206300.0
    1.5Unf      14    76000.0 110150.0000   139400.0


In [28]:
# ==============================================================================
# CÂU 20 – Kết hợp nhiều điều kiện: nhà "đáng mua" (chất lượng cao, giá hợp lý)
# ==============================================================================
# Mục đích : Xác định nhà có OverallQual >= 7, GrLivArea >= 1500,
#            GarageCars >= 2, CentralAir = 'Y' → "nhà đáng mua".
# Kết quả  : Lọc ra những căn nhà tốt nhất về mặt feature để phân tích thêm.
q20 = pysqldf("""
    SELECT
        rowid                      AS id,
        Neighborhood,
        YearBuilt,
        OverallQual,
        ROUND(GrLivArea, 0)        AS dien_tich_song,
        GarageCars,
        CentralAir,
        KitchenQual,
        ROUND(SalePrice, 4)        AS log_price
    FROM df
    WHERE OverallQual >= 7
      AND GrLivArea  >= 1500
      AND GarageCars >= 2
      AND CentralAir = 'Y'
    ORDER BY SalePrice DESC
    LIMIT 15
""")
print("\n[Q20] Danh sách nhà 'đáng mua' (chất lượng cao, đủ tiện nghi):")
print(q20.to_string(index=False))

print("\n" + "="*80)
print("Hoàn thành 20 câu SQL phân tích dữ liệu House Prices!")
print("="*80)


[Q20] Danh sách nhà 'đáng mua' (chất lượng cao, đủ tiện nghi):
  id Neighborhood  YearBuilt  OverallQual  dien_tich_song  GarageCars CentralAir KitchenQual  log_price
 692      NoRidge       1994           10          4316.0           3          Y          Ex   755000.0
1183      NoRidge       1996           10          4476.0           3          Y          Ex   745000.0
1170      NoRidge       1995           10          3627.0           3          Y          Gd   625000.0
 899      NridgHt       2009            9          2364.0           3          Y          Ex   611657.0
 804      NridgHt       2008            9          2822.0           3          Y          Ex   582933.0
1047      StoneBr       2005            9          2868.0           3          Y          Ex   556581.0
 441      NridgHt       2008           10          2402.0           3          Y          Ex   555000.0
 770      StoneBr       2003            8          3279.0           3          Y          Ex   538000.0
